# 🔬 DeepTrace v3 — EfficientNet-B4 Fine-Grained Training Pipeline

This notebook trains an **EfficientNet-B4** model for deepfake detection using the
**[ScaleDF](https://huggingface.co/datasets/WenhaoWang/ScaleDF)** dataset with
**134 fine-grained classes** (46 real sources + 88 fake generation methods).

### 🚨 What changed in v3 (mode-collapse fix)

v2 collapsed: Binary Recall -> `1.000`, Binary Precision froze at `0.650` (exactly the
fake prior of the val set), Fine-Grained Accuracy stuck at ~`1%` (chance = 0.75%).
The model learned the constant function `predict Fake`.

Two independent causes, both fixed here:

| # | Cause | Fix |
|---|-------|-----|
| 1 | **Single-class batches.** ScaleDF stores *one class per `.tar` shard*. v2 used `resampled=True` plus a `.shuffle(2000)` buffer that is filled from **one shard at a time**, so a batch of 16 was ~16 images of the *same* class. Gradients degenerate and BatchNorm statistics thrash, so the net gives up and predicts the majority group. | `ScaleDFTrainDataset` now keeps **24 shard streams open concurrently** and samples across them, so every batch spans ~20+ distinct classes. Slots rotate every 512 samples so all 134 classes get seen. |
| 2 | **Prior imbalance.** 88 fake shards vs 46 real shards means 65.7% fake. "Always Fake" scores 0.650 precision / 1.000 recall / 0.788 F1, and `metric_for_best_model="binary_f1"` actually *rewarded* it, so early stopping never fired. | Sampler draws **50/50 real/fake**, Focal Loss adds **3x class weights on the 46 real classes**, and model selection now uses **balanced accuracy** (collapse scores 0.500). |

### 🧠 Training changes
- **`FocalLossTrainer`**: `Trainer` subclass overriding `compute_loss`
- **Focal Loss** (gamma = 2.0) with per-class weights and mild label smoothing, replacing CrossEntropy
- **Class weights**: real ~ `1.78`, fake ~ `0.59` (mean-normalised, 3:1 ratio), so false positives on real images hurt 3x more
- **LR 3e-5 -> 1e-5** on the backbone, `10x` on the randomly-initialised classifier head
- **Collapse guard**: eval logs `pred_fake_rate`, `real_recall`, `unique_pred_classes` and shouts by step 500 if things are degenerating, instead of you finding out at step 4000

### ⚡ Prerequisites
- Runtime: **T4 GPU** or better
- HuggingFace token as a Kaggle Secret named `HF_TOKEN`

In [ ]:
# -- Module 1: Environment Setup --
# albumentations pinned to 2.0.8 - last MIT-licensed release before AGPL fork.
# opencv-python-headless avoids libGL errors on headless runtimes.
# webdataset streams ScaleDF's tar shards without downloading the full 2TB+ dataset.
!pip install -q transformers datasets accelerate torchvision evaluate scikit-learn \
    "albumentations==2.0.8" opencv-python-headless webdataset huggingface_hub

## 1. Environment, Reproducibility & Authentication

Seeds are fixed for reproducible training. The HuggingFace token is loaded from
Kaggle Secrets (never hardcoded in the notebook).

In [ ]:
# -- Module 2: GPU Detection, Reproducibility & Authentication --
import os
import io
import random
import numpy as np
import torch
from PIL import Image

# -- Reproducibility --
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# -- Device detection --
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU detected. Training will be extremely slow.")

# -- HuggingFace Authentication --
# Try Kaggle Secrets first, then getpass. NEVER hardcode tokens.
try:
    from kaggle_secrets import UserSecretsClient
    _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets.")
except Exception:
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
        print("HF token loaded from Colab Secrets.")
    except Exception:
        import getpass
        _hf_token = getpass.getpass("Enter your HuggingFace token: ")

assert _hf_token, "HF_TOKEN is required to stream ScaleDF."
os.environ["HF_TOKEN"] = _hf_token

# Explicitly authenticate with HF Hub (os.environ alone is not enough)
from huggingface_hub import login
login(token=_hf_token, add_to_git_credential=False)
print("HuggingFace token successfully loaded and authenticated!")

## 2. Load EfficientNet-B4 Processor

The processor carries the exact normalization statistics (`image_mean` / `image_std`) and
input resolution baked into the `google/efficientnet-b4` checkpoint.

> ⚠️ `google/efficientnet-b4`'s `image_std` is `[0.4785, 0.4733, 0.4743]`, **not** the
> generic ImageNet `[0.229, 0.224, 0.225]`. Training with mismatched normalization silently
> degrades every prediction.

In [ ]:
# -- Module 3: Image Processor & Normalization --
from transformers import AutoImageProcessor

MODEL_ID = "google/efficientnet-b4"
processor = AutoImageProcessor.from_pretrained(MODEL_ID)

IMAGE_SIZE = processor.size["height"]  # 380 for B4
NORM_MEAN  = processor.image_mean
NORM_STD   = processor.image_std

print(f"Resolution:     {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Normalize mean: {NORM_MEAN}")
print(f"Normalize std:  {NORM_STD}")

## 3. Fine-Grained 134-Class Label Discovery

ScaleDF organizes its data into WebDataset `.tar` shards:
- **Real** shards have filenames starting with `000000` (e.g., `000000AFAD.tar`, `000000FFHQ.tar`)
- **Fake** shards use generation method names (e.g., `StableDiffusion_faces.tar`, `AMatrix_faces.tar`)

We discover all 134 classes dynamically, build label mappings, and separate
real/fake class IDs for computing binary metrics at eval time.

> **One class per shard.** Remember this - it is exactly why v2 collapsed, and why the
> sampler in Module 6 has to interleave many shards at once.

In [ ]:
# -- Module 4: Fine-Grained 134-Class Label Discovery --
from huggingface_hub import HfApi

api = HfApi()
REPO_ID = "WenhaoWang/ScaleDF"
BASE_URL = "https://huggingface.co/datasets/WenhaoWang/ScaleDF/resolve/main/"
_url_suffix = f"?token={_hf_token}" if _hf_token else ""

# -- Discover training shards --
print("Discovering training shards...")
train_tree = api.list_repo_tree(REPO_ID, path_in_repo="ScaleDF/train", repo_type="dataset")
train_tars = sorted([item.path for item in train_tree if item.path.endswith(".tar")])

# -- Extract class names from shard filenames --
class_names = [os.path.basename(t).replace(".tar", "") for t in train_tars]
NUM_CLASSES = len(class_names)
assert NUM_CLASSES > 0, "No training shards found! Check your HF token and network."

# -- Build label mappings --
label2id = {name: idx for idx, name in enumerate(class_names)}
id2label = {idx: name for idx, name in enumerate(class_names)}

# -- Build binary grouping (for inference-time Real/Fake verdict) --
real_class_ids = sorted([idx for idx, name in enumerate(class_names) if name.startswith("000000")])
fake_class_ids = sorted([idx for idx, name in enumerate(class_names) if not name.startswith("000000")])
assert len(real_class_ids) + len(fake_class_ids) == NUM_CLASSES

REAL_ID_SET = set(real_class_ids)
FAKE_ID_SET = set(fake_class_ids)

# -- Build training URLs and URL-to-label mapping --
train_urls = [BASE_URL + t + _url_suffix for t in train_tars]
url_to_label = {}
for t in train_tars:
    shard_name = os.path.basename(t).replace(".tar", "")
    url_to_label[os.path.basename(t)] = label2id[shard_name]

# -- Print summary --
print(f"\nTotal classes:      {NUM_CLASSES}")
print(f"Real classes:       {len(real_class_ids)}")
print(f"Fake classes:       {len(fake_class_ids)}")
print(f"Training shards:    {len(train_tars)}")

_shard_fake_prior = len(fake_class_ids) / NUM_CLASSES
print(f"\nUniform-shard fake prior: {_shard_fake_prior:.3f}  "
      f"<-- v2's frozen Binary Precision. This is what we are fixing.")
print(f"Sample real classes: {[id2label[i] for i in real_class_ids[:5]]}")
print(f"Sample fake classes: {[id2label[i] for i in fake_class_ids[:5]]}")

## 4. Forensic Augmentation Pipeline

To survive WhatsApp/Instagram compression, we simulate social media degradation
during training. All transforms use the **Albumentations 2.x** API.

In [ ]:
# -- Module 5: Data Augmentation Pipeline --
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(
        translate_percent=(-0.05, 0.05),
        scale=(0.95, 1.05),
        rotate=(-15, 15),
        p=0.5,
    ),
    A.ColorJitter(
        brightness=(0.8, 1.2), contrast=(0.8, 1.2),
        saturation=(0.8, 1.2), hue=(-0.1, 0.1), p=0.5,
    ),
    # Social media compression simulation
    A.ImageCompression(quality_range=(30, 90), p=0.6),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.GaussNoise(std_range=(0.03, 0.12), p=0.3),
    # Partial occlusion (hands, hair, masks)
    A.CoarseDropout(
        num_holes_range=(1, 3),
        hole_height_range=(0.05, 0.15),
        hole_width_range=(0.05, 0.15), p=0.15,
    ),
    A.RandomResizedCrop(size=(IMAGE_SIZE, IMAGE_SIZE), scale=(0.8, 1.0)),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

val_augmentations = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

print("Augmentation pipeline initialized.")

## 5. Streaming Datasets - class-mixed and real/fake balanced

### The v2 bug, precisely

```python
# v2 - every batch was effectively ONE class
wds.WebDataset(train_urls, resampled=True, shardshuffle=True).shuffle(2000)
```

`resampled=True` picks a random shard, then streams it start to finish. The `.shuffle(2000)`
buffer therefore fills from **that single shard**, i.e. a single class. With
`per_device_train_batch_size=16`, every gradient step saw 16 images carrying an identical
label. For a BatchNorm backbone that is close to worst case: batch statistics become
class-conditional, the head only ever sees one-hot targets that drift shard to shard, and
the only stable solution under SGD is a constant prediction. The 88:46 shard ratio decided
*which* constant: `Fake`.

### The v3 sampler

`ScaleDFTrainDataset` keeps `SHARD_POOL_SIZE` (24) shard streams **open at once** - half
pinned to real shards, half to fake - and draws each sample from a randomly chosen slot:

- **~20+ distinct classes per batch of 32**, so gradients are real and BN statistics are healthy
- **50/50 real vs fake** at the sampler level, killing the 0.650 prior before the loss even sees it
- **Slot rotation** every `SLOT_LIFETIME` (512) samples, so all 134 classes get covered rather than only the first 24
- Per-slot shuffle buffers stay small (64 **undecoded** records), so RAM stays flat

Validation now sweeps **all 134 shards** with a 15-sample cap, so fine-grained accuracy is
measured over the complete label set instead of the ~100 classes v2 happened to reach.

In [ ]:
# -- Module 6: Streaming Datasets (class-mixed + balanced) --
import webdataset as wds
from collections import Counter

# -- Sampler configuration --
SHARD_POOL_SIZE   = 24    # concurrent shard streams => distinct classes per batch.
                          # Lower to 12 if Kaggle's network chokes; raise on fast links.
SLOT_LIFETIME     = 512   # samples pulled from one shard before rotating to a new class
REAL_SAMPLE_PROB  = 0.5   # P(draw a real image). 0.5 => balanced, kills the fake prior.
SHARD_SHUFFLE_BUF = 64    # per-slot shuffle of raw records (cheap, pre-decode)


def shard_filename(url):
    """'https://.../StableDiffusion_faces.tar?token=x' -> 'StableDiffusion_faces.tar'"""
    return url.split("?")[0].split("/")[-1]


def extract_image(sample):
    """Extract a PIL Image from a webdataset sample."""
    for key in ("jpg", "jpeg", "png", "webp", "bmp", "tiff", "ppm"):
        if key in sample and isinstance(sample[key], Image.Image):
            return sample[key]
    for key, val in sample.items():
        if key.startswith("__"):
            continue
        if isinstance(val, Image.Image):
            return val
        if isinstance(val, bytes) and len(val) > 100:
            try:
                return Image.open(io.BytesIO(val))
            except Exception:
                continue
    return None


def _validate_image(img):
    """Convert to RGB numpy array, rejecting broken or tiny images."""
    if img is None:
        return None
    try:
        arr = np.array(img.convert("RGB"))
        if arr.ndim != 3 or arr.shape[2] != 3:
            return None
        if min(arr.shape[:2]) < 20:
            return None
        return arr
    except Exception:
        return None


class ScaleDFTrainDataset(torch.utils.data.IterableDataset):
    """Infinite stream of CLASS-MIXED, real/fake-BALANCED samples.

    ScaleDF is one class per .tar shard, so reading shards sequentially (v2) yields
    single-class batches -> mode collapse. This keeps `pool_size` shard readers alive
    simultaneously and samples across them, so a batch spans many classes.

    Slots are partitioned real vs fake and a group is drawn with probability
    `real_prob`, which makes the real/fake mix independent of the 46:88 shard ratio.
    """

    def __init__(self, urls, augmentation, url_to_label, real_id_set,
                 pool_size=SHARD_POOL_SIZE, slot_lifetime=SLOT_LIFETIME,
                 real_prob=REAL_SAMPLE_PROB, seed=SEED):
        super().__init__()
        self.augmentation = augmentation
        self.url_to_label = url_to_label
        self.pool_size = pool_size
        self.slot_lifetime = slot_lifetime
        self.real_prob = real_prob
        self.seed = seed

        self.real_urls, self.fake_urls = [], []
        for u in urls:
            lbl = url_to_label.get(shard_filename(u))
            if lbl is None:
                continue
            (self.real_urls if lbl in real_id_set else self.fake_urls).append(u)

        assert self.real_urls and self.fake_urls, \
            "Need both real and fake shards; check the 000000* prefix convention."
        print(f"Sampler: {len(self.real_urls)} real shards / {len(self.fake_urls)} fake shards, "
              f"pool={pool_size}, P(real)={real_prob}")

    def _open_slot(self, group, rng):
        """Open a fresh stream on a random shard of the requested group."""
        pool = self.real_urls if group == "real" else self.fake_urls
        url = rng.choice(pool)
        pipe = (
            wds.WebDataset(url, shardshuffle=False, handler=wds.warn_and_continue)
            .shuffle(SHARD_SHUFFLE_BUF, handler=wds.warn_and_continue)
            .decode("pil", handler=wds.warn_and_continue)
        )
        return {
            "it": iter(pipe),
            "label": self.url_to_label[shard_filename(url)],
            "left": self.slot_lifetime,
            "group": group,
        }

    def __iter__(self):
        # Distinct RNG per dataloader worker so workers do not mirror each other.
        worker_info = torch.utils.data.get_worker_info()
        wid = worker_info.id if worker_info is not None else 0
        rng = random.Random(self.seed + 9973 * wid)

        n_real = max(1, int(round(self.pool_size * self.real_prob)))
        n_fake = max(1, self.pool_size - n_real)

        slots  = [self._open_slot("real", rng) for _ in range(n_real)]
        slots += [self._open_slot("fake", rng) for _ in range(n_fake)]
        real_idx = list(range(n_real))
        fake_idx = list(range(n_real, n_real + n_fake))

        while True:
            group = "real" if rng.random() < self.real_prob else "fake"
            i = rng.choice(real_idx if group == "real" else fake_idx)

            # Rotate the slot onto a new class once its lifetime expires.
            if slots[i]["left"] <= 0:
                slots[i] = self._open_slot(group, rng)

            try:
                sample = next(slots[i]["it"])
            except StopIteration:
                slots[i] = self._open_slot(group, rng)   # shard exhausted
                continue
            except Exception:
                slots[i] = self._open_slot(group, rng)   # network / tar hiccup
                continue

            slots[i]["left"] -= 1
            label = slots[i]["label"]

            arr = _validate_image(extract_image(sample))
            if arr is None:
                continue
            try:
                pixel_values = self.augmentation(image=arr)["image"]
            except Exception:
                continue

            yield {"pixel_values": pixel_values, "label": label}


class ScaleDFValDataset(torch.utils.data.Dataset):
    """Fixed validation set covering ALL 134 classes.

    Sweeps every shard with a small per-shard cap so fine-grained accuracy is measured
    against the full label space. Stores raw PIL images; val transforms run in __getitem__.
    """

    def __init__(self, urls, url_to_label, augmentation, max_per_shard=15, max_samples=None):
        super().__init__()
        self.augmentation = augmentation
        self.items = []  # list of (PIL.Image, label_id)

        print(f"Collecting val set: {max_per_shard} samples x {len(urls)} shards...")

        for n, url in enumerate(urls):
            if max_samples is not None and len(self.items) >= max_samples:
                break
            label = url_to_label.get(shard_filename(url))
            if label is None:
                continue

            shard_count = 0
            pipe = (
                wds.WebDataset(url, shardshuffle=False, handler=wds.warn_and_continue)
                .decode("pil", handler=wds.warn_and_continue)
            )
            for sample in pipe:
                if shard_count >= max_per_shard:
                    break
                img = extract_image(sample)
                if img is None:
                    continue
                try:
                    rgb = img.convert("RGB")
                    arr = np.array(rgb)
                    if arr.ndim != 3 or arr.shape[2] != 3 or min(arr.shape[:2]) < 20:
                        continue
                    self.items.append((rgb, label))
                    shard_count += 1
                except Exception:
                    continue

            if (n + 1) % 25 == 0:
                print(f"  {n + 1}/{len(urls)} shards -> {len(self.items)} samples")

        dist = Counter(label for _, label in self.items)
        n_real = sum(v for k, v in dist.items() if k in REAL_ID_SET)
        n_fake = sum(v for k, v in dist.items() if k in FAKE_ID_SET)
        print(f"Collected {len(self.items)} val samples "
              f"({n_real} real, {n_fake} fake, {len(dist)}/{NUM_CLASSES} classes covered)")
        missing = NUM_CLASSES - len(dist)
        if missing:
            print(f"  NOTE: {missing} classes returned no usable images.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img, label = self.items[idx]
        arr = np.array(img)
        pixel_values = self.augmentation(image=arr)["image"]
        return {"pixel_values": pixel_values, "label": label}

print("Dataset classes defined.")

In [ ]:
# -- Instantiate Datasets --
# Val is drawn from training shards (same 134 classes) so fine-grained metrics work.
# ScaleDF has 14M+ images, so ~2k val samples is a negligible overlap.

print("Creating training dataset (class-mixed streaming)...")
train_ds = ScaleDFTrainDataset(
    train_urls,
    train_augmentations,
    url_to_label,
    REAL_ID_SET,
    pool_size=SHARD_POOL_SIZE,
    slot_lifetime=SLOT_LIFETIME,
    real_prob=REAL_SAMPLE_PROB,
)

print("\nCreating validation dataset (all 134 shards, 15 per shard)...")
val_ds = ScaleDFValDataset(train_urls, url_to_label, val_augmentations, max_per_shard=15)

assert len(val_ds) > 0, (
    "Validation dataset is empty! Check your network connection and HF token. "
    "If rate-limited, try: huggingface-cli login"
)

# Record the val prior - the number "always Fake" would score. Precision == this => collapse.
VAL_FAKE_PRIOR = float(np.mean([lbl in FAKE_ID_SET for _, lbl in val_ds.items]))
print(f"\nDatasets ready! Val size: {len(val_ds)} | val fake prior: {VAL_FAKE_PRIOR:.3f}")
print(f"Collapse signature to watch for: binary_precision ~= {VAL_FAKE_PRIOR:.3f} with recall 1.000")

## 6. Sanity Check - including **batch class diversity**

The single most important assertion in this notebook: a batch must contain **many
classes and both binary groups**. If `unique classes per batch` comes back as 1-2, the
sampler is broken and no loss function will save the run. Fix it here, not at step 4000.

In [ ]:
# -- Module 7: Sanity Check --
print("Pulling one training sample from the shard stream...")
sample = next(iter(train_ds))
print(f"  pixel_values shape: {sample['pixel_values'].shape}")
print(f"  pixel_values dtype: {sample['pixel_values'].dtype}")

label_id = sample["label"]
print(f"  label ID:       {label_id}")
print(f"  class name:     {id2label[label_id]}")
print(f"  binary group:   {'Fake' if label_id in FAKE_ID_SET else 'Real'}")

assert sample["pixel_values"].shape == (3, IMAGE_SIZE, IMAGE_SIZE), \
    f"Expected (3, {IMAGE_SIZE}, {IMAGE_SIZE}), got {sample['pixel_values'].shape}"

# -- Verify val dataset --
val_sample = val_ds[0]
print(f"\n  val pixel_values shape: {val_sample['pixel_values'].shape}")
print(f"  val class: {id2label[val_sample['label']]}")

# -- CRITICAL: batch diversity check (this is what v2 failed) --
print("\n" + "=" * 62)
print("BATCH CLASS DIVERSITY CHECK")
print("=" * 62)

_probe = torch.utils.data.DataLoader(train_ds, batch_size=32, num_workers=0)
_it = iter(_probe)
_uniques, _fake_rates = [], []
for b in range(4):
    batch = next(_it)
    labels = batch["label"].numpy()
    u = len(set(labels.tolist()))
    fr = float(np.mean([l in FAKE_ID_SET for l in labels]))
    _uniques.append(u)
    _fake_rates.append(fr)
    print(f"  batch {b}: {u:>2}/32 unique classes | fake rate {fr:.2f}")

_mean_u = float(np.mean(_uniques))
_mean_fr = float(np.mean(_fake_rates))
print(f"\n  mean unique classes/batch: {_mean_u:.1f}  (v2 scored ~1.0)")
print(f"  mean fake rate:            {_mean_fr:.2f}  (target ~0.50, v2 was ~0.66)")

assert _mean_u >= 8, (
    f"Batches only span {_mean_u:.1f} classes - the sampler is still degenerate. "
    "Raise SHARD_POOL_SIZE or check that shard streams are opening."
)
assert 0.30 <= _mean_fr <= 0.70, (
    f"Fake rate {_mean_fr:.2f} is off-balance; check REAL_SAMPLE_PROB and the shard split."
)
del _probe, _it

print("\nPipeline is working correctly - batches are class-mixed and balanced.")

In [ ]:
# -- Visualize Training Samples --
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
stream = iter(train_ds)
for ax in axes.flat:
    s = next(stream)
    img = s["pixel_values"].permute(1, 2, 0).numpy()
    img = img * np.array(NORM_STD) + np.array(NORM_MEAN)  # denormalize
    label_name = id2label[s["label"]]
    binary = "Fake" if s["label"] in FAKE_ID_SET else "Real"
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(f"{binary}: {label_name}", fontsize=9)
    ax.axis("off")
plt.suptitle("Augmented Training Samples (134-class, mixed sampler)", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Model & Training - Focal Loss + Class Weights via a custom Trainer

### `FocalLoss`

```
loss = sum_i w[y_i] * (1 - p[y_i])^gamma * CE(x_i, y_i)  /  sum_i w[y_i]
```

- **gamma = 2.0** down-weights easy examples. Once the net is 95% sure, that example's
  gradient shrinks ~400x, so the loss keeps chasing hard fine-grained distinctions instead
  of coasting on the easy "obviously fake" mass that let v2 collapse.
- **w[y]** are per-class weights: the **46 real classes get 3x** the fake classes
  (`REAL_CLASS_BOOST`), mean-normalised to 1.0 so the loss scale (and therefore the
  effective LR) does not move. Guessing Fake on a Real image is now 3x more expensive.
- Loss is computed in **fp32** (`logits.float()`); focal's `(1-p)^gamma` term underflows in fp16.
- Mild **label smoothing (0.05)** lives inside the loss now. `TrainingArguments`'
  `label_smoothing_factor` is deliberately dropped: it is ignored once `compute_loss` is
  overridden, so leaving it in would be a lie.

### Learning rates
Backbone drops **3e-5 -> 1e-5** as requested. But the 134-way head is randomly
initialised, and at 1e-5 it would barely move while the pretrained trunk drifts - a great
way to *cause* a collapse. So the head gets **10x (1e-4)** through a separate param group.
The cosine schedule scales both groups proportionally.

### Selection metric
`binary_f1` was the hidden villain: "always Fake" scores **0.788** F1, so early stopping
happily preserved a dead model. v3 selects on **`binary_balanced_accuracy`** - collapse
scores exactly **0.500** - and a `CollapseGuard` callback shouts at the first eval if
`pred_fake_rate > 0.97` or predictions span fewer than 5 classes. Evals run every **500** steps, so a
bad run is visible in minutes rather than at step 4000.

In [ ]:
# -- Module 8: Model, Focal Loss & Custom Trainer --
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
)
from scipy.special import softmax
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# =====================================================================
# Hyperparameters for the anti-collapse fix
# =====================================================================
FOCAL_GAMMA        = 2.0    # 0 == weighted CE. 2.0 is the Lin et al. default.
REAL_CLASS_BOOST   = 3.0    # weight ratio real:fake. Raise to 4-5 if FPs persist.
LOSS_SMOOTHING     = 0.05   # label smoothing inside the focal loss
BASE_LR            = 1e-5   # backbone (was 3e-5)
HEAD_LR_MULTIPLIER = 10.0   # fresh 134-way head needs a faster LR than the trunk
WEIGHT_DECAY       = 1e-5

# -- Load model with 134-class head --
print(f"Loading EfficientNet-B4 with {NUM_CLASSES} classes...")
model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # Replace default ImageNet head
)
print(f"Model loaded. Output head: {NUM_CLASSES} classes.")

# =====================================================================
# 1. CLASS WEIGHTS - penalize false positives on the 46 real classes
# =====================================================================
_raw = np.array([
    REAL_CLASS_BOOST if i in REAL_ID_SET else 1.0
    for i in range(NUM_CLASSES)
], dtype=np.float32)
# Mean-normalise so the loss magnitude (and the effective LR) is unchanged.
_raw *= NUM_CLASSES / _raw.sum()
CLASS_WEIGHTS = torch.tensor(_raw, dtype=torch.float32)

_w_real = float(CLASS_WEIGHTS[real_class_ids[0]])
_w_fake = float(CLASS_WEIGHTS[fake_class_ids[0]])
print(f"\nClass weights: real={_w_real:.3f} (x{len(real_class_ids)}), "
      f"fake={_w_fake:.3f} (x{len(fake_class_ids)}), mean={CLASS_WEIGHTS.mean():.3f}")
print(f"A false positive (Real -> Fake) now costs {_w_real / _w_fake:.1f}x a false negative.")


# =====================================================================
# 2. FOCAL LOSS - hard-example mining, replaces CrossEntropy
# =====================================================================
class FocalLoss(nn.Module):
    """Multi-class focal loss with per-class weights and label smoothing.

        loss_i = w[y_i] * (1 - p[y_i])^gamma * CE_smoothed(x_i, y_i)

    Normalised by sum(w) rather than N, so upweighting the real classes changes the
    *relative* penalty without inflating the gradient norm.
    """

    def __init__(self, class_weights, gamma=2.0, label_smoothing=0.0, eps=1e-6):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.eps = eps

    def forward(self, logits, targets):
        # fp32: (1-p)^gamma underflows badly under fp16 autocast
        logits = logits.float()
        log_probs = F.log_softmax(logits, dim=-1)

        tgt = targets.view(-1, 1)
        tgt_log_p = log_probs.gather(1, tgt).squeeze(1)
        pt = tgt_log_p.exp().clamp(self.eps, 1.0)

        if self.label_smoothing > 0:
            smooth = -log_probs.mean(dim=-1)
            ce = (1.0 - self.label_smoothing) * (-tgt_log_p) + self.label_smoothing * smooth
        else:
            ce = -tgt_log_p

        focal = (1.0 - pt).pow(self.gamma)
        w = self.class_weights.to(logits.device, logits.dtype)[targets]

        return (w * focal * ce).sum() / w.sum().clamp_min(self.eps)


focal_loss_fn = FocalLoss(
    CLASS_WEIGHTS, gamma=FOCAL_GAMMA, label_smoothing=LOSS_SMOOTHING
)
print(f"\nFocal loss ready: gamma={FOCAL_GAMMA}, smoothing={LOSS_SMOOTHING}")


# =====================================================================
# 3. CUSTOM TRAINER - overrides compute_loss + per-group learning rates
# =====================================================================
class FocalLossTrainer(Trainer):
    """Trainer that swaps HF's built-in CrossEntropy for weighted Focal Loss.

    Also builds the optimizer with three param groups so the randomly-initialised
    classifier head can learn faster (HEAD_LR_MULTIPLIER) than the pretrained trunk,
    and so norm/bias params skip weight decay.
    """

    def __init__(self, *args, loss_fn=None, head_lr_multiplier=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = loss_fn
        self.head_lr_multiplier = head_lr_multiplier

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # **kwargs absorbs num_items_in_batch on transformers >= 4.46
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]
        loss = self.loss_fn(logits, labels)
        inputs["labels"] = labels  # restore for downstream hooks
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        if self.optimizer is not None:
            return self.optimizer

        base_lr = self.args.learning_rate
        decay, no_decay, head = [], [], []
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            if name.startswith("classifier"):
                head.append(param)
            elif param.ndim <= 1 or name.endswith(".bias"):
                no_decay.append(param)   # BN weights + biases: no weight decay
            else:
                decay.append(param)

        groups = [
            {"params": decay,    "lr": base_lr, "weight_decay": self.args.weight_decay},
            {"params": no_decay, "lr": base_lr, "weight_decay": 0.0},
            {"params": head,     "lr": base_lr * self.head_lr_multiplier,
             "weight_decay": self.args.weight_decay},
        ]
        cls, kw = Trainer.get_optimizer_cls_and_kwargs(self.args)
        kw.pop("lr", None)
        self.optimizer = cls(groups, lr=base_lr, **kw)
        print(f"Optimizer: trunk lr={base_lr:.1e} | head lr={base_lr * self.head_lr_multiplier:.1e} "
              f"| {len(decay)} decay / {len(no_decay)} no-decay / {len(head)} head tensors")
        return self.optimizer


# =====================================================================
# 4. METRICS - with explicit collapse diagnostics
# =====================================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.asarray(logits, dtype=np.float32)

    fine_preds = np.argmax(logits, axis=-1)
    fine_acc = float(np.mean(fine_preds == labels))

    probs = softmax(logits, axis=-1)
    fake_probs = probs[:, fake_class_ids].sum(axis=-1)
    binary_preds = (fake_probs > 0.5).astype(int)
    binary_labels = np.isin(labels, fake_class_ids).astype(int)

    prec, rec, f1, _ = precision_recall_fscore_support(
        binary_labels, binary_preds, average="binary", zero_division=0
    )
    acc = accuracy_score(binary_labels, binary_preds)

    # Per-group recall. real_recall == 0.0 is the collapse fingerprint.
    real_mask, fake_mask = binary_labels == 0, binary_labels == 1
    real_recall = float(np.mean(binary_preds[real_mask] == 0)) if real_mask.any() else 0.0
    fake_recall = float(np.mean(binary_preds[fake_mask] == 1)) if fake_mask.any() else 0.0
    balanced_acc = (real_recall + fake_recall) / 2.0

    return {
        "fine_grained_accuracy": fine_acc,
        "binary_accuracy": float(acc),
        "binary_f1": float(f1),
        "binary_precision": float(prec),
        "binary_recall": float(rec),
        # -- collapse diagnostics --
        "binary_balanced_accuracy": balanced_acc,   # 0.500 == degenerate
        "real_recall": real_recall,                 # 0.000 == "everything is fake"
        "fake_recall": fake_recall,
        "pred_fake_rate": float(np.mean(binary_preds)),
        "unique_pred_classes": float(len(np.unique(fine_preds))),
        "mean_fake_prob": float(np.mean(fake_probs)),
    }


class CollapseGuard(TrainerCallback):
    """Shouts at the FIRST eval if the model is going degenerate."""

    def on_evaluate(self, args, state, control, metrics=None, **kw):
        if not metrics:
            return
        fake_rate = metrics.get("eval_pred_fake_rate", 0.0)
        uniq = metrics.get("eval_unique_pred_classes", 0.0)
        bal = metrics.get("eval_binary_balanced_accuracy", 0.0)
        if fake_rate > 0.97 or uniq < 5 or bal < 0.52:
            print("\n" + "!" * 70)
            print(f"!! MODE COLLAPSE WARNING at step {state.global_step}")
            print(f"!!   pred_fake_rate={fake_rate:.3f} | unique_pred_classes={uniq:.0f} "
                  f"| balanced_acc={bal:.3f}")
            print("!!   Try: REAL_CLASS_BOOST 3->5, FOCAL_GAMMA 2->3, "
                  "SHARD_POOL_SIZE 24->32, HEAD_LR_MULTIPLIER 10->20")
            print("!" * 70 + "\n")
        else:
            print(f"  [guard ok] step {state.global_step}: fake_rate={fake_rate:.3f}, "
                  f"balanced_acc={bal:.3f}, classes={uniq:.0f}")


# -- Collate function (prevents Trainer column-pruning crash) --
def collate_fn(examples):
    pixel_values = torch.stack([ex["pixel_values"] for ex in examples])
    labels = torch.tensor([ex["label"] for ex in examples], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}


# =====================================================================
# 5. TRAINING CONFIGURATION
# =====================================================================
OUTPUT_DIR = "./deeptrace-efficientnet-v3"

# T4 supports fp16 only; A100+ supports bf16 (more stable)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Streaming: use max_steps (no epochs)
    max_steps=8000,          # x2 grad accum => ~256k images, same as v2's 15k x bs16

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,   # effective batch 32 -> steadier gradients

    eval_strategy="steps",
    eval_steps=500,          # 2x more often than v2: catch collapse by step 500, not 4000
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    learning_rate=BASE_LR,           # 1e-5 (was 3e-5); head gets 10x via param groups
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    warmup_steps=400,
    max_grad_norm=1.0,
    # NOTE: no label_smoothing_factor - FocalLoss owns smoothing now.

    metric_for_best_model="binary_balanced_accuracy",  # NOT binary_f1: collapse scores 0.788 there
    greater_is_better=True,
    load_best_model_at_end=True,

    fp16=use_fp16,
    bf16=use_bf16,

    # Set to True if you hit CUDA OOM on T4
    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    logging_steps=50,
    dataloader_num_workers=0,        # Prevents Kaggle multiprocessing deadlock
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    report_to="none",
)

trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    loss_fn=focal_loss_fn,
    head_lr_multiplier=HEAD_LR_MULTIPLIER,
    callbacks=[
        CollapseGuard(),
        EarlyStoppingCallback(early_stopping_patience=6),  # evals are 2x more frequent now
    ],
)

print(f"\nTraining for up to {training_args.max_steps:,} steps")
print(f"Effective batch size: "
      f"{training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Eval every {training_args.eval_steps:,} steps | selecting on binary_balanced_accuracy")
print(f"Mixed precision: {'bf16' if use_bf16 else 'fp16' if use_fp16 else 'none'}")
print(f"Loss: FocalLoss(gamma={FOCAL_GAMMA}, real_boost={REAL_CLASS_BOOST}, "
      f"smoothing={LOSS_SMOOTHING})")

In [ ]:
# -- Baseline sanity: what does an UNTRAINED head score? --
# Fine-grained ~0.75% (1/134) and balanced accuracy ~0.50 are EXPECTED here.
# The point is the reference line: if the step-500/1000 evals still look like this, stop and tune.
print("Evaluating the untrained 134-class head as a baseline...")
_baseline = trainer.evaluate()
for k in ("eval_fine_grained_accuracy", "eval_binary_balanced_accuracy",
          "eval_binary_precision", "eval_binary_recall",
          "eval_pred_fake_rate", "eval_unique_pred_classes"):
    if k in _baseline:
        print(f"  {k:34s} {_baseline[k]:.4f}")

In [ ]:
# Run Training
# To resume after a disconnect, change to: trainer.train(resume_from_checkpoint=True)
#
# Healthy signs by step 500-1500:
#   pred_fake_rate        drifting toward ~0.50-0.65 (NOT pinned at 1.000)
#   real_recall           > 0.30 and climbing
#   unique_pred_classes   > 20
#   fine_grained_accuracy climbing past 5-10%
# If pred_fake_rate is still 1.000 at step 1500, CollapseGuard will tell you what to bump.
print("Starting EfficientNet-B4 fine-grained training on ScaleDF...")
trainer.train()

## 8. Evaluation

Fine-grained (134-class) and binary (Real vs Fake) metrics, confusion matrix, and a
**threshold sweep**. Summing 88 fake-class probabilities against 46 real ones is
structurally biased toward Fake, so the optimal binary cut is rarely 0.5 - we find it
here and ship it to the backend in `class_metadata.json`.

In [ ]:
# -- Module 9: Evaluation --
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

print("Evaluating on validation set...")
metrics = trainer.evaluate()
for k, v in sorted(metrics.items()):
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Detailed predictions
predictions = trainer.predict(val_ds)
logits = np.asarray(predictions.predictions, dtype=np.float32)
y_pred_fine = np.argmax(logits, axis=-1)
y_true = predictions.label_ids

probs = softmax(logits, axis=-1)
fake_score = probs[:, fake_class_ids].sum(axis=-1)
y_true_binary = np.isin(y_true, fake_class_ids).astype(int)

# -- Threshold sweep on the summed fake probability --
print("\n" + "=" * 62)
print("BINARY THRESHOLD SWEEP (optimising balanced accuracy)")
print("=" * 62)
BEST_THRESHOLD, _best_bal = 0.5, -1.0
for t in np.arange(0.05, 0.96, 0.05):
    bal = balanced_accuracy_score(y_true_binary, (fake_score > t).astype(int))
    flag = ""
    if bal > _best_bal:
        BEST_THRESHOLD, _best_bal, flag = float(t), bal, "  <-- best"
    print(f"  threshold {t:.2f}: balanced_acc {bal:.4f}{flag}")
print(f"\nChosen threshold: {BEST_THRESHOLD:.2f} (balanced accuracy {_best_bal:.4f})")

y_pred_binary = (fake_score > BEST_THRESHOLD).astype(int)

print("\n" + "=" * 62)
print(f"BINARY CLASSIFICATION REPORT (Real vs Fake @ {BEST_THRESHOLD:.2f})")
print("=" * 62)
print(classification_report(
    y_true_binary, y_pred_binary,
    target_names=["Real", "Fake"], digits=4, zero_division=0
))

cm = confusion_matrix(y_true_binary, y_pred_binary, labels=[0, 1])
print("Binary Confusion Matrix (rows=true, cols=predicted):")
print(f"{'':>8}{'Real':>8}{'Fake':>8}")
for i, name in enumerate(["Real", "Fake"]):
    print(f"{name:>8}" + "".join(f"{v:>8}" for v in cm[i]))

fn = cm[1][0]  # Fake classified as Real
fp = cm[0][1]  # Real classified as Fake
print(f"\nFalse negatives (Fake -> Real): {fn} / {cm[1].sum()}")
print(f"False positives (Real -> Fake): {fp} / {cm[0].sum()}")

# -- Collapse verdict --
print("\n" + "=" * 62)
print("COLLAPSE CHECK")
print("=" * 62)
_uniq = len(np.unique(y_pred_fine))
_fake_rate = float(np.mean(y_pred_binary))
print(f"  unique predicted classes: {_uniq} / {NUM_CLASSES}")
print(f"  predicted-fake rate:      {_fake_rate:.3f} (val prior {VAL_FAKE_PRIOR:.3f})")
print(f"  balanced accuracy:        {_best_bal:.4f}")
if _fake_rate > 0.97 or _uniq < 5:
    print("  STATUS: STILL COLLAPSED - bump REAL_CLASS_BOOST / FOCAL_GAMMA and rerun Module 8.")
else:
    print("  STATUS: healthy - the model is discriminating, not guessing.")

# -- Fine-grained accuracy --
top5 = np.mean([
    y_true[i] in np.argsort(logits[i])[-5:] for i in range(len(y_true))
])
print(f"\nFine-grained Top-1 accuracy: {np.mean(y_pred_fine == y_true):.4f}  (chance {1/NUM_CLASSES:.4f})")
print(f"Fine-grained Top-5 accuracy: {top5:.4f}")

## 9. Save, Verify & Export

Saves the trained model + processor + class metadata. A round-trip reload
verifies the checkpoint is valid before exporting.

> `class_metadata.json` now also carries **`binary_threshold`** (from the sweep above) and
> the loss configuration. The backend must use that threshold instead of a hardcoded `0.5`,
> otherwise it re-introduces the fake-side bias at inference time.

In [ ]:
# -- Module 10: Save, Verify & Export --
import json
import shutil

# 1. Save model + processor
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# 2. Save class metadata for the backend
class_metadata = {
    "num_classes": NUM_CLASSES,
    "real_class_ids": real_class_ids,
    "fake_class_ids": fake_class_ids,
    "id2label": {str(k): v for k, v in id2label.items()},
    "label2id": label2id,
    # Backend: fake_prob = probs[fake_class_ids].sum(); verdict = fake_prob > binary_threshold
    "binary_threshold": float(globals().get("BEST_THRESHOLD", 0.5)),
    "training": {
        "version": "v3-focal",
        "loss": "focal",
        "focal_gamma": FOCAL_GAMMA,
        "real_class_boost": REAL_CLASS_BOOST,
        "label_smoothing": LOSS_SMOOTHING,
        "base_lr": BASE_LR,
        "head_lr_multiplier": HEAD_LR_MULTIPLIER,
        "sampler": {
            "shard_pool_size": SHARD_POOL_SIZE,
            "slot_lifetime": SLOT_LIFETIME,
            "real_sample_prob": REAL_SAMPLE_PROB,
        },
    },
}
with open(os.path.join(OUTPUT_DIR, "class_metadata.json"), "w") as f:
    json.dump(class_metadata, f, indent=2)
print(f"Saved class_metadata.json (binary_threshold="
      f"{class_metadata['binary_threshold']:.2f})")

# 3. Round-trip verification
print("\nVerifying saved checkpoint...")
_check_model = AutoModelForImageClassification.from_pretrained(OUTPUT_DIR)
_check_processor = AutoImageProcessor.from_pretrained(OUTPUT_DIR)

assert _check_model.config.num_labels == NUM_CLASSES, \
    f"Expected {NUM_CLASSES} labels, got {_check_model.config.num_labels}"
assert len(_check_model.config.id2label) == NUM_CLASSES, \
    f"Expected {NUM_CLASSES} id2label entries, got {len(_check_model.config.id2label)}"
assert list(_check_processor.image_mean) == list(NORM_MEAN), \
    f"Norm mean mismatch: saved={_check_processor.image_mean}, expected={NORM_MEAN}"
assert list(_check_processor.image_std) == list(NORM_STD), \
    f"Norm std mismatch: saved={_check_processor.image_std}, expected={NORM_STD}"

print(f"  num_labels:  {_check_model.config.num_labels}")
print(f"  norm mean:   {_check_processor.image_mean}")
print(f"  norm std:    {_check_processor.image_std}")
print("  Checkpoint verified!")
del _check_model, _check_processor

# 4. Zip and download
zip_filename = "deeptrace_efficientnet_v3"
shutil.make_archive(zip_filename, "zip", OUTPUT_DIR)
print(f"\nCreated {zip_filename}.zip")

try:
    from google.colab import files
    files.download(f"{zip_filename}.zip")
    print("Downloading via browser...")
except Exception:
    print("Running on Kaggle: Download the zip from the 'Output' tab.")

print("\nExtract into backend/models/deeptrace-efficientnet-v3 and update DEEPTRACE_MODEL in .env!")
print("Remember: the backend must read binary_threshold from class_metadata.json.")

## 10. If it *still* collapses - tuning ladder

Work down this list one change at a time, and judge at step 500-1500 (not 4000):

1. **`REAL_CLASS_BOOST` 3.0 -> 5.0** - the direct lever on false positives. Above ~8 the
   model starts over-predicting Real, so watch `fake_recall`.
2. **`FOCAL_GAMMA` 2.0 -> 3.0** - more aggressive hard-example focus. Above 3 the gradient
   gets noisy on a 134-way head.
3. **`SHARD_POOL_SIZE` 24 -> 32** and **`SLOT_LIFETIME` 512 -> 256** - more classes per batch,
   faster class rotation. Costs network throughput.
4. **`HEAD_LR_MULTIPLIER` 10 -> 20** - if `unique_pred_classes` stays low, the head simply
   is not moving at 1e-5.
5. **Warm up the head first** - freeze the trunk for ~1000 steps
   (`for p in model.efficientnet.parameters(): p.requires_grad = False`), then unfreeze.
   Cleanest fix when a fresh head is dragging the pretrained features down.
6. **Two-stage curriculum** - train a 2-class binary head to convergence, then re-init to
   134 classes on top of those features. Slower, but the most reliable route to real
   fine-grained accuracy.

> Do **not** raise the backbone LR back to 3e-5 while chasing this. v2's problem was never
> too little learning; it was a degenerate data distribution plus a loss that rewarded the
> lazy answer.